# Week 4 · Day 10 — Prompt Engineering for Business Applications

**Course:** IPAM USL 5-Week Short Course: Introduction to Artificial Intelligence *(Introductory tier)*

**Facilitator:** Solomon Wilson MBCS | PhD Student, Computer Science | Deputy HOD Transport Planning & Operations | HOD, IT & Audit Supervisor, SLPTA

**Mode:** Google Colab (zero-install)

**Mental model layer:** L10 — Prompting as Specification

**Running scenario:** Route **R12** (Wilberforce → CBD) — operator OP-104, 25-minute delay

**Module:** 2 · **Week:** 4 · **Tier:** Intro

**New concept:** A prompt is a specification — the model does what you describe, not what you mean

**Deliverable wired in:** None

## Learning objectives
By the end of today you will be able to:
- Improve answers with **few-shot** examples and clear **system instructions**.
- Ask the model for **structured output** (JSON) you can use in code.
- Apply prompting to real SLPTA tasks: complaint routing, incident records, and a Krio passenger SMS.

## Why this matters for SLPTA

On Day 9 we got Gemini to answer. Today we make it answer **reliably and in a usable shape**. A prompt is really a **specification**: the clearer and more example-driven it is, the more dependable the output. Reliable structure is what lets us drop an LLM into a real dispatch workflow.

## Environment setup

In [ ]:
# google-genai is REQUIRED today.
!pip install -q google-genai
print("Environment ready.")

In [ ]:
# --- Standard SLPTA bootstrap (identical in every notebook) ----------------
import sys
from pathlib import Path
for candidate in [Path.cwd(), *Path.cwd().parents,
                  Path("/content/IPAM_USL_Intro_AI_5Week")]:
    if (candidate / "shared" / "slpta_bootstrap.py").exists():
        sys.path.insert(0, str(candidate / "shared"))
        break

from slpta_bootstrap import (MODEL, ensure_course_data, get_client,
                             load_route12_context, load_route_logs,
                             load_complaints, load_routes, load_operators)

ensure_course_data()
print("Model configured:", MODEL)
print(load_route12_context())

## API key reminder
This notebook calls Gemini, so you need your key. In Colab: click the **key icon** (Secrets) in the left sidebar, add a secret named **`GEMINI_API_KEY`**, paste your key, and toggle notebook access on.

<!-- cell-diagram:c22 -->
<p align="center"></p>

### Check your understanding (before running)
This cell runs: `import time`

**Predict:** Before you run it, what do you expect the output to show for Route **R12** (Wilberforce → CBD, OP-104, 25-minute delay)? Write your guess.

In [ ]:
import time

# --- Gemini API Configuration ---
# Establish a connection to the Gemini API using the course-provided bootstrap helper.
client = get_client()

def ask(prompt):
    """
    Sends a text prompt to Gemini with a built-in retry mechanism for rate limits.

    Parameters:
    -----------
    prompt : str
        The instructions or query to be processed by the LLM.

    Returns:
    --------
    str
        The plain-text response generated by the model.
    """
    max_retries = 5
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(model=MODEL, contents=prompt)
            return response.text
        except Exception as e:
            # If we hit a rate limit (429), we need a significant pause on the free tier
            if "429" in str(e) and attempt < max_retries - 1:
                # Wait 60s, then 120s, etc. to ensure quota reset
                wait_time = 60 * (attempt + 1)
                print(f"Quota limit reached. Waiting {wait_time}s to reset... (Attempt {attempt+1}/{max_retries})")
                time.sleep(wait_time)
                continue
            raise e

print(f"Connected to {MODEL} with robust auto-retry enabled.")

## Concept — first principles

- **Zero-shot**: you ask with no examples.
- **Few-shot**: you include 2–3 worked examples; the model copies the pattern.
- **System instruction**: a sentence that sets role and rules ("You are an SLPTA dispatch assistant…").
- **Structured output**: you demand a fixed shape (e.g. JSON) so your *code* can read the answer, not just a human.

*Jargon, defined once:* **few-shot** = giving the model a few example input→output pairs inside the prompt.

<p align="center"></p>

<!-- cell-diagram:c22 -->
<p align="center"></p>

### Check your understanding (before running)
We will classify complaints two ways: zero-shot, then few-shot. **Predict:** will adding three labelled examples make the categories more consistent, or make no difference?

In [ ]:
import time

# 1. Load the dataset and sample 5 random complaints for testing
complaints = load_complaints().sample(5, random_state=11)["raw_text"].tolist()

# --- Zero-shot Classification ---
def classify_zero_shot(text):
    """
    Sends a single complaint to the model with instructions to categorize it.
    """
    prompt = (
        "Classify this SLPTA complaint into ONE of: Delay, Overcharging, Safety, "
        "Cleanliness, Staff Conduct, Lost Item, Other. Reply with only the category.\n\n"
        + text
    )
    return ask(prompt).strip()

# Iterate through our sample and print the classification results
print(f"{'CATEGORY':14s} | {'COMPLAINT TEXT'}")
print("-" * 70)
for c in complaints:
    try:
        category = classify_zero_shot(c)
        print(f"{category:14s} <- {c[:55]}...")
        # Use a longer delay (10s) to respect strict free-tier rate limits
        time.sleep(10)
    except Exception as e:
        print(f"Error: {e}")
        break

> **If you see a 429 error:** The free Gemini tier has a per-minute request limit.
> Wait 60 seconds and re-run the cell. This is normal — your API key is fine.
> The `ask()` function already retries automatically, but if your daily quota is exhausted,
> get a fresh API key from **aistudio.google.com**.

<!-- cell-diagram:c13 -->
<p align="center"></p>

### Check your understanding (before running)
Few-shot examples in the prompt show Gemini how to label complaints.

**Predict:** Will accuracy improve vs zero-shot when you add 3 labelled R12 examples?

In [ ]:
# --- Few-shot Classification ---
# Providing examples (shots) helps the model understand the expected format and nuance.
FEWSHOT_PROMPT = (
    "You are an SLPTA dispatch assistant. Classify each complaint into ONE of: "
    "Delay, Overcharging, Safety, Cleanliness, Staff Conduct, Lost Item, Other. "
    "Reply with only the category.\n\n"
    "Complaint: Bus on R7 came 40 minutes late.\nCategory: Delay\n\n"
    "Complaint: Driver was overtaking dangerously.\nCategory: Safety\n\n"
    "Complaint: I left my umbrella on the R3 bus.\nCategory: Lost Item\n\n"
)

def classify_few_shot(text):
    """
    Uses a prompt with examples to categorize the complaint.
    """
    return ask(FEWSHOT_PROMPT + "Complaint: " + text + "\nCategory:").strip()

print(f"{'CATEGORY':14s} | {'COMPLAINT TEXT'}")
print("-" * 70)
for c in complaints:
    try:
        category = classify_few_shot(c)
        print(f"{category:14s} <- {c[:55]}...")
        # Wait 10 seconds between requests to avoid quota exhaustion
        time.sleep(10)
    except Exception as e:
        print(f"Error: {e}")
        break

### Demo — structured output (JSON) we can use in code

### Check your understanding (before running)
We will ask Gemini to extract structured fields from the R12 incident report and return them as **JSON**.

**Predict:** The model will return plain text by default. Do you think it will actually produce valid JSON, or will it add extra sentences around the data? What will it write for `delay_minutes`?

*Write your prediction, then run.*

In [ ]:
import json

# 1. Access the raw incident report file
# ensure_course_data() provides the path to our local storage
data_path = ensure_course_data()
incident_file = data_path / "incident_reports" / "incident_R12_2025-02-14.txt"
incident_text = incident_file.read_text()

# 2. Request Structured Output (JSON) from Gemini
# We specify the exact keys we want to ensure the program can reliably read the data.
raw_response = ask(
    "From the incident report, return ONLY valid JSON (no markdown fences) with keys: "
    "route, operator, delay_minutes, cause. Use only facts in the report.\n\n" + incident_text
)

# 3. Post-process the raw text string
# LLMs often add formatting (like ```json ... ```) that prevents direct JSON parsing.
# We clean the response to isolate the raw JSON string.
clean_json = raw_response.strip().strip("`")
if clean_json.lower().startswith("json"):
    clean_json = clean_json[4:].strip()

try:
    # 4. Convert the string into a Python Dictionary (record)
    record = json.loads(clean_json)

    # Display the structured data
    print("Parsed record:", record)

    # Demonstrate programmatically accessing a specific data field
    print(f"We can now use a field directly -> Route: {record['route']}")

except json.JSONDecodeError as e:
    print(f"Failed to parse JSON. Raw output was: {raw_response}")
    print(f"Error: {e}")

### Demo — a passenger SMS in Krio
SLPTA serves passengers who read **Krio**. The model can draft in local language — a real localisation win.

<!-- cell-diagram:c19 -->
<p align="center"></p>

### Check your understanding (before running)
We will ask Gemini to draft a passenger SMS in **Krio** about the R12 delay.

**Predict:** Do you think Gemini will produce accurate Krio, a mix of Krio and English, or mostly English? Will it keep the message under 160 characters without being told the character count?

*Write your prediction, then run.*

In [ ]:
# --- Passenger Notification in Krio ---
# Draft a message for local passengers using Sierra Leonean Krio.
def generate_krio_sms():
    """
    Generates a passenger advisory SMS in Krio language.
    """
    prompt = (
        "Write a short, polite passenger SMS in Krio about Route R12 running 25 minutes "
        "late due to heavy traffic. Apologise. Keep it under 160 characters."
    )

    # The 'ask' function now handles retries internally if the API is busy
    sms_content = ask(prompt)
    return sms_content

sms = generate_krio_sms()
print("Draft SMS (Krio):")
print("-" * 20)
print(sms)
print("-" * 20)
print(f"Length: {len(sms)} characters")

### Exercise — improve a prompt (change one thing)

Replace `___FILL_TONE___` with a tone word (e.g. `formal`, `friendly`, `urgent`) and re-run to see how tone changes the SMS.

<!-- cell-diagram:c22 -->
<p align="center"></p>

### Check your understanding (before running)
You will change one word in the prompt — the **tone** — and ask Gemini to redraft the R12 passenger SMS.

**Before filling in `tone`:** Predict how a `formal` tone will differ from a `friendly` tone in the output. Will the model change vocabulary, sentence length, or both?

*Write your prediction, fill in `tone`, then run.*

In [ ]:
import time

# --- Exercise: Tone Modification ---
# The goal of this exercise is to observe how a single 'tone' keyword changes the LLM's output.

# TODO: Replace '___FILL_TONE___' with a specific tone (e.g., 'formal', 'reassuring', or 'apologetic').
tone = "___FILL_TONE___"

def generate_toned_sms(selected_tone):
    """
    Constructs and sends a prompt to Gemini to generate a notification with a specific tone.
    """
    prompt = (
        f"Write a {selected_tone} passenger SMS (under 160 chars) about Route R12 running "
        "25 minutes late due to heavy traffic. Apologise."
    )
    # The 'ask' function (defined earlier) handles the API call and exponential backoff.
    return ask(prompt)

if "___FILL_TONE___" in tone:
    print("Action Required: Please set a tone in the 'tone' variable above, then re-run this cell.")
else:
    print(f"Generating SMS with a {tone} tone...")
    try:
        output = generate_toned_sms(tone)
        print("\nGenerated SMS:")
        print("-" * 20)
        print(output)
        print("-" * 20)
    except Exception as e:
        print(f"An error occurred: {e}")

## Check your understanding
1. What is the difference between zero-shot and few-shot prompting?
2. Why did we ask for **JSON** in the extraction demo instead of a sentence?
3. Give one SLPTA task where drafting in Krio (or another local language) would matter.

### Your answers
*Double-click to edit this cell and type your answers here.*

1.
2.
3.

## If you remember one thing today…

> **A prompt is a specification — be explicit about the format and show examples, and the output becomes reliable enough to use.**

## Submission checklist
- [ ] Run every code cell successfully (top to bottom)
- [ ] Complete the exercise (fill every blank / make the requested change)
- [ ] Answer the **Check your understanding** questions in the markdown cell provided
- [ ] Save a clean copy of the notebook (*File → Save a copy in Drive*)